# Automated CMOS Pattern Extraction

Use this notebook to test the headless order-pattern extraction helpers in `echelle_spectra.tools.pattern_extraction`.

The workflow mirrors `02_pattern_calibration.ipynb`, but the repeatable parts now live in package code:

1. Load sphere and background frames.
2. Build a background-subtracted detector image.
3. Detect order centers in sampled detector columns.
4. Fit polynomial traces for all orders.
5. Compare the proposed pattern against the current reference pattern.
6. Save only after visual review.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import echelle_spectra
from echelle_spectra.tools.echelle import Calibrations
from echelle_spectra.tools.pattern_extraction import (
    PatternExtractionConfig,
    extract_order_pattern,
    subtract_background,
)

%matplotlib inline

## Calibration Inputs

The default files are the packaged 2024 CMOS sphere/background and pattern files. For a new calibration date, change `files_cmos["sphr"]`, `files_cmos["bkgr"]`, and `OUTPUT_PATTERN` before running the notebook.


In [ ]:
CALIB_DIR = echelle_spectra._config["base_path"] / "resources/calibration_files"

files_cmos = {
    "orders": "pattern_CMOS_20240305.txt",
    "wavelength": "Th_wavelength_CMOS_20240305.txt",
    "sphr": "sphere_cmos_20240305.sif",
    "bkgr": "sphere_cmos_20240305_bkg.sif",
    "integral": "integrating_sphere.txt",
}

OUTPUT_PATTERN = "pattern_CMOS_NEWDATE.txt"
REFERENCE_PATTERN = CALIB_DIR / files_cmos["orders"]

CALIB_DIR

## Load Sphere And Background

`Calibrations.load_sphere()` keeps the same image-loading path used by existing extraction notebooks. The automation starts after frames are loaded.


In [ ]:
cb = Calibrations(folder=str(CALIB_DIR), filenames=files_cmos)
cb.load_sphere()

image = subtract_background(cb.sphr.images, cb.bkgr.images)
sphere_mean = cb.sphr.images.mean(axis=0)
background_mean = cb.bkgr.images.mean(axis=0)

print("Sphere stack:", cb.sphr.images.shape)
print("Background stack:", cb.bkgr.images.shape)
print("Subtracted image:", image.shape)
print("Detector size from calibration:", (cb.DIMO, cb.DIMW))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharex=True, sharey=True)
for ax, data, title in zip(
    axes,
    [sphere_mean, background_mean, image],
    ["Sphere mean", "Background mean", "Sphere - background"],
):
    positive = data[np.isfinite(data) & (data > 0)]
    vmin = max(float(np.percentile(positive, 1)), 1.0) if positive.size else 1.0
    vmax = float(np.percentile(positive, 99.8)) if positive.size else 1.0
    ax.imshow(data, origin="lower", cmap="inferno", norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("Column px")
axes[0].set_ylabel("Row px")
plt.tight_layout()

## Extraction Parameters

Start with the same CMOS detection parameters as the manual notebook. The packaged 2024 sphere frame has one spurious peak near the left edge, so this notebook uses explicit review columns beginning at `680 px`. For a new dataset, adjust `SAMPLE_COLUMNS`, `peak_threshold`, or `peak_min_dist_px` until every sampled column reports 29 peaks.


In [ ]:
config = PatternExtractionConfig(
    expected_orders=29,
    sample_step_px=150,
    sample_count=10,
    smooth_window_px=21,
    smooth_polyorder=1,
    amplification_rate=3e-3,
    peak_threshold=0.13,
    peak_min_dist_px=50,
    baseline_poly_deg=6,
    trace_poly_degree=2,
)

SAMPLE_COLUMNS = np.arange(680, 680 + 10 * 150, 150)
result = extract_order_pattern(image, config=config, columns_px=SAMPLE_COLUMNS)

print("Pattern shape:", result.pattern.shape)
print("Sampled columns:", result.columns_px.tolist())
for detection in result.detections:
    status = "OK" if detection.n_peaks == config.expected_orders else "CHECK"
    print(f"column {detection.column_px:4d}: {detection.n_peaks:2d} peaks  {status}")

## Peak Detection Review

The red markers are detected order centers at the sampled columns. The white lines are the fitted order traces evaluated across the whole detector.


In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
positive = sphere_mean[np.isfinite(sphere_mean) & (sphere_mean > 0)]
norm = mcolors.LogNorm(vmin=max(float(np.percentile(positive, 1)), 1.0), vmax=float(np.percentile(positive, 99.8)))
ax.imshow(sphere_mean, origin="lower", cmap="viridis", norm=norm, aspect="auto")

x = np.arange(result.pattern.shape[0])
for order_idx in range(result.n_orders):
    ax.plot(x, result.pattern[:, order_idx], "w-", lw=0.8, alpha=0.75)

for detection in result.detections:
    ax.plot(
        np.full(detection.n_peaks, detection.column_px),
        detection.row_peaks_px,
        ".r",
        markersize=4,
    )

ax.set_title("Automated order-pattern overlay")
ax.set_xlabel("Column px")
ax.set_ylabel("Row px")
plt.tight_layout()

## Compare Against Reference Pattern

This does not decide whether the new pattern is correct. It shows how much the newly fitted traces differ from the current `pattern_CMOS_20240305.txt` reference.


In [ ]:
reference_pattern = np.loadtxt(REFERENCE_PATTERN, dtype=int)
if reference_pattern.shape != result.pattern.shape:
    raise ValueError(f"Reference shape {reference_pattern.shape} differs from new shape {result.pattern.shape}")

delta = result.pattern.astype(float) - reference_pattern.astype(float)

print("Delta row px: new - reference")
print("  median:", float(np.median(delta)))
print("  mean:  ", float(np.mean(delta)))
print("  std:   ", float(np.std(delta)))
print("  min/max:", float(np.min(delta)), float(np.max(delta)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

im = axes[0].imshow(delta.T, origin="lower", aspect="auto", cmap="coolwarm")
axes[0].set_title("Trace delta: new - reference")
axes[0].set_xlabel("Column px")
axes[0].set_ylabel("Order index")
plt.colorbar(im, ax=axes[0], label="Row px")

axes[1].plot(np.arange(result.n_orders), np.median(delta, axis=0), "o-", label="median")
axes[1].fill_between(
    np.arange(result.n_orders),
    np.percentile(delta, 10, axis=0),
    np.percentile(delta, 90, axis=0),
    alpha=0.25,
    label="10-90%",
)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_title("Per-order delta summary")
axes[1].set_xlabel("Order index")
axes[1].set_ylabel("Row px")
axes[1].legend()
plt.tight_layout()

## Optional Save

Leave `SAVE = False` until the overlay and reference comparison look physically reasonable. Use a dated filename and keep historical pattern files intact.


In [ ]:
SAVE = False
save_path = CALIB_DIR / OUTPUT_PATTERN

if SAVE:
    np.savetxt(save_path, result.pattern, fmt="%d")
    print(f"Saved {save_path}")
else:
    print(f"SAVE=False; proposed pattern not written. Target would be: {save_path}")